In [1]:
!pip install pyspark -q


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ETL_Aggregate") \
    .getOrCreate()

print("✅ Spark démarré :", spark.version)

✅ Spark démarré : 4.0.2


In [4]:
df = spark.read.parquet('/content/drive/MyDrive/ETL_projet/silver/')

print("✅ Silver chargé :", df.count(), "lignes")

✅ Silver chargé : 779425 lignes


In [5]:
from pyspark.sql.functions import sum, round

gold_pays = df.groupBy('Country') \
    .agg(round(sum('TotalPrice'), 2).alias('CA_total')) \
    .orderBy('CA_total', ascending=False)

gold_pays.write.mode("overwrite").parquet(
    '/content/drive/MyDrive/ETL_projet/gold/ventes_par_pays/'
)

print("✅ Gold ventes_par_pays sauvegardé !")
gold_pays.show(5)

✅ Gold ventes_par_pays sauvegardé !
+--------------+-------------+
|       Country|     CA_total|
+--------------+-------------+
|United Kingdom|1.438923492E7|
|          EIRE|    616570.54|
|   Netherlands|    554038.09|
|       Germany|    425019.71|
|        France|    348768.96|
+--------------+-------------+
only showing top 5 rows


In [6]:
from pyspark.sql.functions import month, year, sum, round

gold_mois = df.withColumn('Mois', month('InvoiceDate')) \
    .withColumn('Annee', year('InvoiceDate')) \
    .groupBy('Annee', 'Mois') \
    .agg(round(sum('TotalPrice'), 2).alias('CA_mensuel')) \
    .orderBy('Annee', 'Mois')

gold_mois.write.mode("overwrite").parquet(
    '/content/drive/MyDrive/ETL_projet/gold/ventes_par_mois/'
)

print("✅ Gold ventes_par_mois sauvegardé !")
gold_mois.show(5)

✅ Gold ventes_par_mois sauvegardé !
+-----+----+----------+
|Annee|Mois|CA_mensuel|
+-----+----+----------+
| 2009|  12| 683504.01|
| 2010|   1| 555802.67|
| 2010|   2| 504558.96|
| 2010|   3| 696978.47|
| 2010|   4|  591982.0|
+-----+----+----------+
only showing top 5 rows


In [7]:
from pyspark.sql.functions import sum, round

gold_produits = df.groupBy('Description') \
    .agg(round(sum('TotalPrice'), 2).alias('CA_total')) \
    .orderBy('CA_total', ascending=False) \
    .limit(10)

gold_produits.write.mode("overwrite").parquet(
    '/content/drive/MyDrive/ETL_projet/gold/top_produits/'
)

print("✅ Gold top_produits sauvegardé !")
gold_produits.show(10)

✅ Gold top_produits sauvegardé !
+--------------------+---------+
|         Description| CA_total|
+--------------------+---------+
|REGENCY CAKESTAND...|277656.25|
|WHITE HANGING HEA...|247048.01|
|PAPER CRAFT , LIT...| 168469.6|
|              Manual|151777.67|
|JUMBO BAG RED RET...|134307.44|
|             POSTAGE|124648.04|
|ASSORTED COLOUR B...|124351.86|
|       PARTY BUNTING|103283.38|
|MEDIUM CERAMIC TO...| 81416.73|
|PAPER CHAIN KIT 5...| 76598.18|
+--------------------+---------+

